# 🌊 Week 3, Day 2 — June 2, 2026
## Aggregate Plastic Pollution Records into Grid Cells



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw": BASE_DIR+"/data/raw","processed": BASE_DIR+"/data/processed",
        "metadata": BASE_DIR+"/data/metadata","visualizations": BASE_DIR+"/visualizations"}
LAT_MIN,LAT_MAX,LON_MIN,LON_MAX = 5,30,65,95
YEAR_MIN,YEAR_MAX = 2015,2026

In [ ]:
df_plastic = pd.read_csv(DIRS['processed']+'/plastic_clean_final.csv', parse_dates=['Date'])
print(f'Plastic records loaded: {len(df_plastic):,}')
print(df_plastic.columns.tolist())

## Step 1: Assign Grid Coordinates

In [ ]:
# grid_lat = floor(Latitude), grid_lon = floor(Longitude)
# This assigns each coordinate to its containing 1° cell
df_plastic['grid_lat'] = np.floor(df_plastic['Latitude']).astype(int)
df_plastic['grid_lon'] = np.floor(df_plastic['Longitude']).astype(int)
df_plastic['year']     = pd.to_datetime(df_plastic['Date']).dt.year.astype(int)
df_plastic['month']    = pd.to_datetime(df_plastic['Date']).dt.month.astype(int)

print('Grid assignment complete:')
print(f'  Unique grid cells covered: {df_plastic[["grid_lat","grid_lon"]].drop_duplicates().shape[0]}')
print(f'  Year range in data: {df_plastic["year"].min()}–{df_plastic["year"].max()}')
print()
print('Sample:')
print(df_plastic[['Latitude','Longitude','grid_lat','grid_lon','year','month','Plastic_Density_kg_km2']].head(5).to_string())

## Step 2: Aggregate by Grid Cell

In [ ]:
# For each grid cell + year + month, sum up plastic density
# and count records, and collect unique plastic types
plastic_grid = df_plastic.groupby(['grid_lat','grid_lon','year','month']).agg(
    Plastic_Density_kg_km2 = ('Plastic_Density_kg_km2', 'sum'),
    Plastic_Record_Count   = ('Plastic_Weight_kg',       'count'),
    Plastic_Types_Present  = ('Plastic_Type',            lambda x: '|'.join(sorted(x.unique())))
).reset_index()

print(f'Plastic aggregated grid: {len(plastic_grid):,} rows')
print(f'  Columns: {plastic_grid.columns.tolist()}')
print()
print('Top 10 highest density grid cells:')
top = plastic_grid.nlargest(10,'Plastic_Density_kg_km2')
print(top[['grid_lat','grid_lon','year','month','Plastic_Density_kg_km2']].to_string(index=False))

## Step 3: Annotate with Ocean Zone

In [ ]:
def assign_zone(lat, lon):
    if   5 <= lat <= 25 and 65 <= lon <= 75: return 'Arabian_Sea'
    elif 5 <= lat <= 22 and 75 <= lon <= 90: return 'Bay_of_Bengal'
    elif 10<= lat <= 25 and 90 <= lon <= 95: return 'Andaman_Sea'
    elif 5 <= lat <= 12 and 72 <= lon <= 80: return 'Lakshadweep_Gulf_of_Mannar'
    elif 20<= lat <= 30 and 65 <= lon <= 75: return 'Gulf_of_Kutch'
    else: return 'Indian_Ocean'

plastic_grid['Ocean_Zone'] = plastic_grid.apply(
    lambda r: assign_zone(r['grid_lat'], r['grid_lon']), axis=1)

print('Zone distribution in plastic grid:')
print(plastic_grid['Ocean_Zone'].value_counts())

In [ ]:
plastic_grid.to_csv(DIRS['processed']+'/plastic_grid_aggregated.csv', index=False)
print(f'Saved plastic_grid_aggregated.csv ({len(plastic_grid):,} rows) ✅')